# Day 10: Numerical Architectures & Design Trade-offs

## Pre-Class Videos (~55 minutes total)

| # | Segment | Duration | File |
|---|---------|----------|------|
| 1 | Timing & Constraints Essentials | ~15 min | `d10_s1_timing_essentials.html` |
| 2 | Numerical Architecture Trade-offs | ~20 min | `d10_s2_numerical_architectures.html` |
| 3 | PPA — Performance, Power, Area | ~15 min | `d10_s3_ppa_intro.html` |
| 4 | Open-Source ASIC PPA (Aspirational) | ~5 min | `d10_s4_asic_ppa_context.html` |

## Code Examples

| File | Description |
|------|-------------|
| `code/day10_adder_widths.v` | Behavioral adder at 4/8/16/32-bit widths for LUT scaling demo |
| `code/day10_mult_widths.v` | Combinational multiplier at 4/8/16-bit widths for LUT explosion demo |

## Diagrams

| File | Description |
|------|-------------|
| `diagrams/d10_critical_path.svg` | Critical path: FF A → combinational logic → FF B with timing equation |
| `diagrams/d10_pipeline_fix.svg` | Before/after pipelining: long chain → parallel stages |
| `diagrams/d10_ppa_triangle.svg` | PPA trade-off triangle: Performance ↔ Power ↔ Area |

## Key Concepts
- Setup time, hold time, critical path, slack (positive = pass, negative = fail)
- `--freq 25` on nextpnr enables timing analysis
- Adder architectures: ripple-carry (O(N)) vs CLA (O(log N)) vs behavioral `+` (tool-chosen)
- Multiplier: combinational `*` = O(N²) LUTs on iCE40 (no DSP blocks!)
- Sequential shift-and-add: O(N) LUTs, N cycles latency — area vs. latency trade-off
- Fixed-point Q-format: Q4.4 × Q4.4 = Q8.8, extract bits carefully
- PPA proxies: Fmax (performance), LUT/FF count (area), toggle rate (power)
- Structured PPA reporting: comparison tables + written analysis
- Design-space exploration: synthesize at multiple parameters, plot curves
- OpenROAD/OpenLane: same Verilog, real ASIC PPA metrics

## Directory Structure

```
lectures/week3_day10/
├── d10_s1_timing_essentials.html
├── d10_s2_numerical_architectures.html
├── d10_s3_ppa_intro.html
├── d10_s4_asic_ppa_context.html
├── code/
│   ├── day10_adder_widths.v
│   └── day10_mult_widths.v
├── diagrams/
│   ├── d10_critical_path.svg
│   ├── d10_pipeline_fix.svg
│   └── d10_ppa_triangle.svg
├── day10_quiz.md
└── day10_readme.md
```

## Changes from Previous Version

This is a **major content realignment**. The previous version of Day 10 lectures
covered Timing, Clocking & Constraints (PLLs, CDC) as 4 full segments. That content
has been condensed into Segment 1 (~15 min), and the remaining 3 segments now cover
the curriculum-specified topics: numerical architectures, PPA introduction, and ASIC
context. PLL and CDC exercises are available as stretch content in the lab.

---
## Code Examples

### `day10_adder_widths.v`

```verilog
// =============================================================================
// day10_adder_widths.v — Behavioral Adder at Multiple Widths
// Day 10: Numerical Architectures & Design Trade-offs
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Synthesize each module independently to compare LUT scaling:
//   yosys -p "read_verilog day10_adder_widths.v; synth_ice40 -top adder_4bit;  stat"
//   yosys -p "read_verilog day10_adder_widths.v; synth_ice40 -top adder_8bit;  stat"
//   yosys -p "read_verilog day10_adder_widths.v; synth_ice40 -top adder_16bit; stat"
//   yosys -p "read_verilog day10_adder_widths.v; synth_ice40 -top adder_32bit; stat"
// =============================================================================

module adder_4bit (
    input  wire [3:0]  i_a, i_b,
    output wire [4:0]  o_sum
);
    assign o_sum = i_a + i_b;
endmodule

module adder_8bit (
    input  wire [7:0]  i_a, i_b,
    output wire [8:0]  o_sum
);
    assign o_sum = i_a + i_b;
endmodule

module adder_16bit (
    input  wire [15:0] i_a, i_b,
    output wire [16:0] o_sum
);
    assign o_sum = i_a + i_b;
endmodule

module adder_32bit (
    input  wire [31:0] i_a, i_b,
    output wire [32:0] o_sum
);
    assign o_sum = i_a + i_b;
endmodule
```

### `day10_mult_widths.v`

```verilog
// =============================================================================
// day10_mult_widths.v — Combinational Multiplier at Multiple Widths
// Day 10: Numerical Architectures & Design Trade-offs
// Accelerated HDL for Digital System Design · UCF ECE
// =============================================================================
// Synthesize each module independently to see LUT explosion:
//   yosys -p "read_verilog day10_mult_widths.v; synth_ice40 -top mult_4bit;  stat"
//   yosys -p "read_verilog day10_mult_widths.v; synth_ice40 -top mult_8bit;  stat"
//   yosys -p "read_verilog day10_mult_widths.v; synth_ice40 -top mult_16bit; stat"
//
// Compare: adder LUTs grow linearly; multiplier LUTs grow quadratically.
// iCE40 has NO DSP blocks — every * is pure LUT logic.
// =============================================================================

module mult_4bit (
    input  wire [3:0] i_a, i_b,
    output wire [7:0] o_product
);
    assign o_product = i_a * i_b;
endmodule

module mult_8bit (
    input  wire [7:0]  i_a, i_b,
    output wire [15:0] o_product
);
    assign o_product = i_a * i_b;
endmodule

module mult_16bit (
    input  wire [15:0] i_a, i_b,
    output wire [31:0] o_product
);
    assign o_product = i_a * i_b;
endmodule
```

---
## Pre-Class Self-Check Quiz

**Q1:** What does positive slack mean? What about negative slack? What happens in hardware with negative slack?

<details><summary>Answer</summary>
Positive slack = timing met (data arrives before the setup time deadline). Negative slack = timing violation — the design may work intermittently, capturing wrong values on some clock cycles. This causes the worst kind of bugs: random, non-reproducible failures.
</details>

**Q2:** Why does `assign product = a * b;` use so many LUTs on the iCE40? Would it be different on a larger FPGA?

<details><summary>Answer</summary>
The iCE40 has **no DSP blocks**. Multiplication is implemented entirely in LUT logic, with O(N²) gate count growth. On larger FPGAs (Xilinx, Intel, larger Lattice), the `*` operator maps to dedicated DSP slices — nearly free in LUTs.
</details>

**Q3:** When would you choose a sequential (shift-and-add) multiplier over a combinational one? Explain using PPA.

<details><summary>Answer</summary>
When **area** is constrained and **latency** is acceptable. The sequential version uses O(N) LUTs but takes N clock cycles. The combinational version uses O(N²) LUTs but completes in 1 cycle. Additionally, the sequential version has a shorter critical path, so it can run at a higher Fmax. Trade-off: area + Fmax vs. latency.
</details>

**Q4:** What are the three FPGA PPA proxies, and which tools measure them?

<details><summary>Answer</summary>
**Performance** → Fmax from `nextpnr` timing report (`--freq` flag). **Area** → LUT count, FF count, EBR count from `yosys stat`. **Power** → estimated from toggle rate × capacitance (limited on iCE40, mostly conceptual). The iCE40 HX1K has 1,280 LUTs, 1,280 FFs, 16 EBR blocks, and 1 PLL.
</details>

**Q5:** In Q4.4 fixed-point format, what is the product of 2.5 × 3.0? How many bits is the full product, and which bits do you extract for a Q4.4 result?

<details><summary>Answer</summary>
2.5 × 3.0 = **7.5**. The full product of two Q4.4 numbers is Q8.8 (16 bits). To extract Q4.4, take bits [11:4] — the 4 lower integer bits and 4 upper fractional bits. The integer part (7) is in bits [11:8], and 0.5 is in bits [7:4] = 4'b1000.
</details>